# 🍽️ RECIPE RECOMMENDATION SYSTEM

This notebook implements a recipe recommendation system using various machine learning techniques including TF-IDF for text vectorization, One-Hot Encoding for categorical features, MinMax Scaling for numerical features, PCA for dimensionality reduction, KMeans for clustering, and KNN for classification. It also leverages Cosine Similarity and Euclidean Distance for finding similar recipes.

### Topics Covered:
*   TF-IDF
*   Cosine Similarity
*   Euclidean Distance
*   K-Nearest Neighbors (KNN)
*   Principal Component Analysis (PCA)
*   K-Means Clustering

## 1. Imports & Setup

This section imports all the necessary libraries for data manipulation, feature engineering, dimensionality reduction, clustering, classification, and similarity computations.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing           import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.metrics.pairwise        import cosine_similarity, euclidean_distances
from sklearn.neighbors               import KNeighborsClassifier
from sklearn.decomposition           import PCA
from sklearn.cluster                 import KMeans
from sklearn.model_selection         import cross_val_score
from scipy.sparse                    import hstack, csr_matrix

## 2. Load Data

The `load_data` function is responsible for loading the recipe dataset from a CSV file, performing initial data cleaning such as dropping null values, converting text to lowercase, and stripping whitespace.

In [2]:
def load_data(filepath="ml_recipes.csv"):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Dataset not found: {filepath}")

    df = pd.read_csv(filepath)[["name", "ingredients", "cuisine", "time"]]
    df.dropna(inplace=True)
    df["name"]        = df["name"].str.lower().str.strip()
    df["ingredients"] = df["ingredients"].str.lower().str.strip()
    df["cuisine"]     = df["cuisine"].str.lower().str.strip()
    df.reset_index(drop=True, inplace=True)

    print(f"[INFO] Loaded {len(df)} recipes | {df['cuisine'].nunique()} cuisines")
    return df

In [3]:
# Load the dataset
df = load_data("ml_recipes.csv")

# Display the first few rows of the DataFrame
display(df.head())

[INFO] Loaded 1000 recipes | 9 cuisines


,name,ingredients,cuisine,time
0,tomato curry,tomato onion butter garlic,indian,30
1,pasta alfredo,pasta cream cheese garlic,italian,25
2,chicken tacos,chicken tortilla onion cilantro lime,mexican,30
3,beef fried rice,rice beef egg soy onion carrot,chinese,20
4,pad thai,noodles shrimp egg peanut bean-sprout lime,thai,25


## 3. Feature Engineering

The `create_features` function transforms raw data into a numerical format suitable for machine learning models using three main techniques: TF-IDF for text, One-Hot Encoding for categorical data, and MinMax Scaling for numerical data.

In [4]:
def create_features(df):
    # TF-IDF on ingredients (text vectorization)
    tfidf = TfidfVectorizer(max_features=200)
    ing_matrix = tfidf.fit_transform(df["ingredients"])

    # One-Hot Encoding for cuisine
    ohe = OneHotEncoder(handle_unknown="ignore")
    cuisine_matrix = ohe.fit_transform(df[["cuisine"]])

    # MinMax Normalization for cooking time
    scaler = MinMaxScaler()
    time_scaled = scaler.fit_transform(df[["time"]])
    time_sparse  = csr_matrix(time_scaled)

    # Combine all into one feature matrix
    features = hstack([ing_matrix, cuisine_matrix, time_sparse])
    print(f"[INFO] Feature matrix: {features.shape}")
    return features, tfidf, ohe, scaler

### Techniques Explained:

*   **TF-IDF (Term Frequency-Inverse Document Frequency)**:
    *   **Purpose**: Converts text data (ingredients) into a numerical representation, highlighting words that are important to a document but not common across all documents.
    *   **Implementation**: `TfidfVectorizer` from `sklearn.feature_extraction.text` is used.
*   **One-Hot Encoding**:
    *   **Purpose**: Converts categorical data (cuisine types) into a binary (0 or 1) vector representation, preventing the model from assuming ordinal relationships between categories.
    *   **Implementation**: `OneHotEncoder` from `sklearn.preprocessing` is used.
*   **MinMax Normalization**:
    *   **Purpose**: Scales numerical features (cooking time) to a fixed range, typically 0 to 1, to prevent features with larger values from dominating the learning process.
    *   **Implementation**: `MinMaxScaler` from `sklearn.preprocessing` is used.

In [5]:
# Create the feature matrix
features, tfidf, ohe, scaler = create_features(df)

[INFO] Feature matrix: (1000, 210)


## 4. PCA — Dimensionality Reduction

Principal Component Analysis (PCA) is used to reduce the dimensionality of the feature space while retaining as much variance as possible. This helps in speeding up subsequent machine learning algorithms and mitigating the curse of dimensionality.

In [6]:
def apply_pca(features):
    dense = features.toarray()
    n_comp = min(50, dense.shape[0], dense.shape[1])
    pca = PCA(n_components=n_comp, random_state=42)
    reduced = pca.fit_transform(dense)
    var = sum(pca.explained_variance_ratio_) * 100
    print(f"[INFO] PCA: {n_comp} components, {var:.1f}% variance retained")
    return reduced, pca

In [7]:
# Apply PCA to the feature matrix
reduced, pca = apply_pca(features)

[INFO] PCA: 50 components, 79.5% variance retained


## 5. KMeans Clustering

KMeans clustering is an unsupervised learning algorithm that partitions data points into K distinct clusters. Here, it groups recipes based on their features into a number of clusters equal to the number of unique cuisines. This helps in discovering inherent groupings within the recipe dataset.

In [8]:
def apply_kmeans(reduced, df):
    n_clusters = df["cuisine"].nunique()   # driven by data, not hardcoded
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df = df.copy()
    df["cluster"] = kmeans.fit_predict(reduced)
    print(f"[INFO] KMeans: {n_clusters} clusters")
    return df, kmeans

In [9]:
# Apply KMeans clustering
df, kmeans = apply_kmeans(reduced, df)

# Display the DataFrame with new cluster assignments
display(df.head())

[INFO] KMeans: 9 clusters


,name,ingredients,cuisine,time,cluster
0,tomato curry,tomato onion butter garlic,indian,30,8
1,pasta alfredo,pasta cream cheese garlic,italian,25,6
2,chicken tacos,chicken tortilla onion cilantro lime,mexican,30,4
3,beef fried rice,rice beef egg soy onion carrot,chinese,20,3
4,pad thai,noodles shrimp egg peanut bean-sprout lime,thai,25,1


## 6. KNN Classifier — Predict Cuisine + Cross-Val Accuracy

A K-Nearest Neighbors (KNN) classifier is trained to predict the cuisine of a recipe based on its features. Cross-validation is used to evaluate the model's performance and ensure its generalization ability.

In [10]:
def train_knn(reduced, df):
    le = LabelEncoder()
    y  = le.fit_transform(df["cuisine"])

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(reduced, y)

    # Cross-validation accuracy (5-fold)
    scores = cross_val_score(knn, reduced, y, cv=5, scoring="accuracy")
    print(f"[INFO] KNN 5-Fold Accuracy: {scores.mean()*100:.2f}% +/- {scores.std()*100:.2f}%")

    return knn, le

In [11]:
# Train the KNN classifier
knn, le = train_knn(reduced, df)

[INFO] KNN 5-Fold Accuracy: 100.00% +/- 0.00%


## 7. Similarity Matrices

Two similarity metrics are computed: Cosine Similarity and Euclidean Distance. These metrics are fundamental for identifying how similar one recipe is to another in the feature space.

*   **Cosine Similarity**: Measures the cosine of the angle between two non-zero vectors. A value closer to 1 indicates higher similarity, while 0 indicates orthogonality (no similarity), and -1 indicates complete dissimilarity. It's often used for text data because it's sensitive to orientation rather than magnitude.
*   **Euclidean Distance**: Measures the straight-line distance between two points in Euclidean space. A smaller distance indicates higher similarity. It's a common distance metric, but sensitive to the scale of features.

In [12]:
def compute_similarity(features):
    cos_sim  = cosine_similarity(features)
    euc_dist = euclidean_distances(features)
    return cos_sim, euc_dist

In [13]:
# Compute similarity matrices
cos_sim, euc_dist = compute_similarity(features)

## 8. Recommend by Recipe Name

This function takes a recipe name as input and recommends other recipes based on their similarity (using both Cosine Similarity and Euclidean Distance) to the input recipe.

In [14]:
def recommend_by_name(name, df, cos_sim, euc_dist):
    name = name.lower().strip()
    matches = df[df["name"] == name]

    if matches.empty:
        partial = df[df["name"].str.contains(name, na=False)]
        if partial.empty:
            print(f"[!] Recipe '{name}' not found.")
            return
        idx = partial.index[0]
        print(f"[INFO] Showing results for: '{df.iloc[idx]['name']}'")
    else:
        idx = matches.index[0]

    cos_scores = sorted(enumerate(cos_sim[idx]),  key=lambda x: x[1], reverse=True)[1:6]
    euc_scores = sorted(enumerate(euc_dist[idx]), key=lambda x: x[1])[1:6]

    print(f"\n=== Top 5 by Cosine Similarity ===")
    for i, score in cos_scores:
        r = df.iloc[i]
        print(f"  {r['name']:<30} cuisine: {r['cuisine']:<12} score: {score:.4f}")

    print(f"\n=== Top 5 by Euclidean Distance ===")
    for i, dist in euc_scores:
        r = df.iloc[i]
        print(f"  {r['name']:<30} cuisine: {r['cuisine']:<12} dist:  {dist:.4f}")

In [15]:
# Example: Recommend recipes similar to 'chicken curry'
# recommend_by_name("chicken curry", df, cos_sim, euc_dist)

## 9. Custom Recommendation (ingredients + cuisine + time)

This function allows users to get recommendations by providing a custom set of ingredients, desired cuisine, and cooking time. It uses the trained models (TF-IDF, One-Hot Encoder, MinMax Scaler, PCA, KNN, KMeans) to process the custom input and find the most similar recipes.

In [16]:
def recommend_custom(ingredients, cuisine, time,
                     df, features, tfidf, ohe, scaler,
                     knn, le, pca, kmeans):

    ing_vec     = tfidf.transform([ingredients.lower()])
    cuisine_vec = ohe.transform([[cuisine.lower()]])
    time_vec    = scaler.transform([[float(time)]])

    query = hstack([ing_vec, cuisine_vec, csr_matrix(time_vec)])

    # Cosine similarity against all recipes
    scores = cosine_similarity(query, features).flatten()
    top    = scores.argsort()[::-1][:5]

    # PCA transform for KNN + KMeans
    query_pca    = pca.transform(query.toarray())
    pred_cuisine = le.inverse_transform(knn.predict(query_pca))[0]
    cluster      = kmeans.predict(query_pca)[0]

    print(f"\n=== Custom Recommendation ===")
    print(f"  Predicted Cuisine : {pred_cuisine}")
    print(f"  Cluster Group     : {cluster}")
    print(f"\n  Top 5 Matches:")
    for i in top:
        r = df.iloc[i]
        print(f"  {r['name']:<30} cuisine: {r['cuisine']:<12} score: {scores[i]:.4f}")

## 10. Interactive Recommendation Menu

This section provides an interactive command-line interface to allow users to either recommend recipes by name or get custom recommendations based on ingredients, cuisine, and cooking time. To exit the interactive menu, type `3`.

In [18]:
# Main interactive loop
print("\n" + "="*50)
print("   RECIPE RECOMMENDATION SYSTEM READY")
print("="*50)

while True:
    print("\n  1. Recommend by recipe name")
    print("  2. Custom recommendation (ingredients + cuisine + time)")
    print("  3. Exit")
    ch = input("\nEnter choice: ").strip()

    if ch == "1":
        name = input("Enter recipe name: ")
        recommend_by_name(name, df, cos_sim, euc_dist)

    elif ch == "2":
        ing   = input("Enter ingredients (space separated): ")
        cuis  = input("Enter cuisine (e.g. indian, italian): ")
        time  = input("Enter cooking time (minutes): ")
        recommend_custom(ing, cuis, time,
                         df, features, tfidf, ohe, scaler,
                         knn, le, pca, kmeans)

    elif ch == "3":
        print("Have a Great Meal!")
        break
    else:
        print("[!] Invalid choice. Enter 1, 2, or 3.")


   RECIPE RECOMMENDATION SYSTEM READY

  1. Recommend by recipe name
  2. Custom recommendation (ingredients + cuisine + time)
  3. Exit

Enter choice: 1
Enter recipe name: paneer tikka

=== Top 5 by Cosine Similarity ===
  paneer bhurji                  cuisine: indian       score: 0.8557
  kadai paneer                   cuisine: indian       score: 0.7724
  doi ilish                      cuisine: indian       score: 0.7645
  fish tikka                     cuisine: indian       score: 0.7462
  prawn moilee                   cuisine: indian       score: 0.7280

=== Top 5 by Euclidean Distance ===
  paneer bhurji                  cuisine: indian       dist:  0.7679
  kadai paneer                   cuisine: indian       dist:  0.9670
  doi ilish                      cuisine: indian       dist:  0.9836
  fish tikka                     cuisine: indian       dist:  1.0212
  prawn moilee                   cuisine: indian       dist:  1.0552

  1. Recommend by recipe name
  2. Custom recomme